# 🎥 ROTBOTS — Video Generator
## From Prompts to Video Clips

**Workshop: "Anatomy of a Content Machine"**  
LI-MA Transformation Digital Art 2026, Amsterdam

---

This notebook takes the `prompts.json` from the Script Writer notebook and:

1. **Generate** — Send each prompt to fal.ai to create video clips
2. **Found Footage** — Optionally upload your own video clips instead of (or alongside) AI-generated ones
3. **Preview** — Watch and compare all clips before assembly

The output (video files) will be used in the Assembly notebook.

> **🤔 Think about:** What does it cost (financially, environmentally) to generate these clips?  
> How does AI-generated footage compare to "real" footage? Can you tell the difference?

---

## 🔧 Setup

In [ ]:
# ============================================================
# SETUP — Run this cell first!
# ============================================================

!pip install -q fal-client requests

import fal_client
import json
import os
import time
import requests
from pathlib import Path
from IPython.display import display, Markdown, HTML, Video

# Working directory (must match Script Writer notebook)
WORK_DIR = Path("rotbots_output")
WORK_DIR.mkdir(exist_ok=True)
VIDEOS_DIR = WORK_DIR / "videos"
VIDEOS_DIR.mkdir(exist_ok=True)

print("✅ Dependencies installed")
print(f"📁 Working directory: {WORK_DIR}")
print(f"🎬 Videos directory: {VIDEOS_DIR}")

In [ ]:
# ============================================================
# API KEY — Paste your fal.ai API key here
# Get one at: https://fal.ai/dashboard/keys
# ============================================================

FAL_API_KEY = ""  # <-- Paste your key between the quotes

if not FAL_API_KEY:
    print("⚠️  Please paste your fal.ai API key above!")
    print("   Get one at: https://fal.ai/dashboard/keys")
else:
    os.environ["FAL_KEY"] = FAL_API_KEY
    print("✅ fal.ai API key configured")

In [ ]:
# ============================================================
# LOAD PROMPTS from Script Writer notebook
# ============================================================

prompts_file = WORK_DIR / "prompts.json"

if prompts_file.exists():
    with open(prompts_file) as f:
        prompts = json.load(f)
    print(f"✅ Loaded {len(prompts)} prompts from {prompts_file}")
    for p in prompts:
        print(f"   Scene {p['scene']}: {p['title']} [{p['style']}] ({p['duration']}s)")
else:
    print("❌ No prompts.json found!")
    print("   Run the Script Writer notebook first, or upload prompts.json to rotbots_output/")

---
## 🎬 Station 5: Generate Videos

Choose your video model and generate clips for each scene.

Each clip takes **1-4 minutes** to generate. While you wait, think about:

> **🤔 Discussion:**
> - Each 5-second clip costs roughly $0.25-$0.50. A 30-second video = ~$2-3 in API costs.
> - The GPU running this model consumes significant electricity. What's the carbon footprint?
> - Who owns this generated footage? You? The model creators? The training data sources?

In [ ]:
# ============================================================
# VIDEO GENERATION SETTINGS
# ============================================================

# Choose your model (swap the endpoint string to try different models)
VIDEO_MODELS = {
    "wan-2.2": {
        "endpoint": "fal-ai/wan/v2.2/1.3b/text-to-video",
        "description": "Wan 2.2 1.3B — Fast, good quality, cheapest (~$0.05/s)",
        "default_seconds": 5,
    },
    "minimax-standard": {
        "endpoint": "fal-ai/minimax/video-01",
        "description": "MiniMax Hailuo 01 — Higher quality, ~$0.28/video",
        "default_seconds": 6,
    },
    "ltx-2.3": {
        "endpoint": "fal-ai/ltx-video/v2.3",
        "description": "LTX 2.3 — Fast, good prompt adherence",
        "default_seconds": 5,
    },
}

# ⬇️ CHOOSE YOUR MODEL HERE ⬇️
CHOSEN_MODEL = "wan-2.2"  # Options: wan-2.2, minimax-standard, ltx-2.3

model = VIDEO_MODELS[CHOSEN_MODEL]
print(f"🎥 Model: {CHOSEN_MODEL}")
print(f"   {model['description']}")
print(f"   Endpoint: {model['endpoint']}")
print(f"\n   Estimated cost for {len(prompts)} scenes: ~${len(prompts) * 0.25:.2f} - ${len(prompts) * 0.50:.2f}")

In [ ]:
# ============================================================
# GENERATE VIDEOS — This takes a few minutes per scene!
# ============================================================

print(f"🎬 Generating {len(prompts)} video clips with {CHOSEN_MODEL}...")
print(f"   This will take approximately {len(prompts) * 2}-{len(prompts) * 4} minutes.")
print("=" * 60)

generated_videos = []

for i, prompt_data in enumerate(prompts):
    scene_num = prompt_data["scene"]
    t2v_prompt = prompt_data["t2v_prompt"]
    duration = prompt_data.get("duration", model["default_seconds"])

    print(f"\n🎥 Scene {scene_num} ({i+1}/{len(prompts)}): {prompt_data['title']}")
    print(f"   Style: {prompt_data['style']}")
    print(f"   Prompt: {t2v_prompt[:100]}...")
    print(f"   Generating ({duration}s clip)...", end=" ", flush=True)

    start_time = time.time()

    try:
        # Build the input based on model
        input_data = {"prompt": t2v_prompt}

        # Some models support duration/num_frames
        if CHOSEN_MODEL == "wan-2.2":
            input_data["num_frames"] = duration * 16 + 1  # WAN uses 16fps
            input_data["aspect_ratio"] = "16:9"
            input_data["enable_prompt_expansion"] = False
        elif CHOSEN_MODEL == "ltx-2.3":
            input_data["num_frames"] = duration * 24 + 1
            input_data["aspect_ratio"] = "16:9"

        # Subscribe (handles polling automatically)
        result = fal_client.subscribe(
            model["endpoint"],
            arguments=input_data,
        )

        elapsed = time.time() - start_time

        # Extract video URL
        video_url = result.get("video", {}).get("url", "")
        if not video_url:
            # Some models return differently
            video_url = result.get("url", result.get("output", {}).get("url", ""))

        if video_url:
            # Download the video
            video_path = VIDEOS_DIR / f"scene_{scene_num:03d}.mp4"
            resp = requests.get(video_url, timeout=120)
            with open(video_path, "wb") as f:
                f.write(resp.content)

            file_size = video_path.stat().st_size / 1024
            generated_videos.append({
                "scene": scene_num,
                "title": prompt_data["title"],
                "path": str(video_path),
                "url": video_url,
                "duration": duration,
                "source": "generated"
            })
            print(f"✅ Done! ({elapsed:.0f}s, {file_size:.0f}KB)")
        else:
            print(f"⚠️ No video URL in response")
            print(f"   Response: {json.dumps(result)[:200]}")

    except Exception as e:
        elapsed = time.time() - start_time
        print(f"❌ Error after {elapsed:.0f}s: {e}")

print(f"\n{'=' * 60}")
print(f"✅ Generated {len(generated_videos)}/{len(prompts)} videos")
print(f"📁 Saved to: {VIDEOS_DIR}")

---
## 📂 Found Footage (Optional)

You can upload your own video clips to use instead of — or alongside — AI-generated ones.

This lets you:
- **Mix real and generated footage** for artistic contrast
- **Skip video generation** if you want to save API costs
- **Explore the uncanny valley** between real and synthetic video

Upload MP4 files using the Colab file browser (📁 icon on the left), or run the cell below to use Colab's upload widget.

In [ ]:
# ============================================================
# UPLOAD FOUND FOOTAGE (Optional)
# ============================================================

USE_FOUND_FOOTAGE = False  # Set to True to upload your own clips

if USE_FOUND_FOOTAGE:
    from google.colab import files
    print("📂 Upload your video files (MP4):")
    print("   Name them scene_002.mp4, scene_003.mp4 etc. to replace specific scenes")
    print("   Or use any name — they'll be added as extra footage")
    uploaded = files.upload()

    for filename, content in uploaded.items():
        dest = VIDEOS_DIR / filename
        with open(dest, "wb") as f:
            f.write(content)
        print(f"   ✅ Saved: {dest} ({len(content)/1024:.0f}KB)")

        # Track in our video list
        scene_num = 0
        try:
            # Try to extract scene number from filename
            scene_num = int(''.join(filter(str.isdigit, filename.split('.')[0])))
        except ValueError:
            scene_num = 100 + len(generated_videos)  # Append at end

        generated_videos.append({
            "scene": scene_num,
            "title": f"Found Footage: {filename}",
            "path": str(dest),
            "duration": 0,  # Unknown until we probe it
            "source": "found_footage"
        })
else:
    print("ℹ️ Found footage disabled. Set USE_FOUND_FOOTAGE = True to upload your own clips.")

---
## 👀 Preview Videos

Watch all generated clips before moving to the Assembly notebook.

In [ ]:
# ============================================================
# PREVIEW ALL VIDEOS
# ============================================================

print(f"🎬 Preview: {len(generated_videos)} video clips")
print("=" * 60)

# Sort by scene number
generated_videos.sort(key=lambda x: x["scene"])

for v in generated_videos:
    video_path = v["path"]
    source_tag = "🤖 AI" if v["source"] == "generated" else "📂 Found"
    display(Markdown(f"### Scene {v['scene']}: {v['title']} — {source_tag}"))

    if Path(video_path).exists():
        try:
            display(Video(video_path, embed=True, width=640))
        except Exception:
            print(f"   (Cannot preview inline — file at: {video_path})")
    else:
        print(f"   ❌ File not found: {video_path}")

    display(Markdown("---"))

In [ ]:
# ============================================================
# SAVE VIDEO MANIFEST — For the Assembly notebook
# ============================================================

manifest = {
    "model": CHOSEN_MODEL,
    "videos": generated_videos
}

manifest_file = WORK_DIR / "video_manifest.json"
with open(manifest_file, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"✅ Video manifest saved to {manifest_file}")
print(f"   {len(generated_videos)} clips ready for assembly")
print(f"\n📁 Files in {VIDEOS_DIR}:")
for f in sorted(VIDEOS_DIR.glob("*.mp4")):
    size = f.stat().st_size / 1024
    print(f"   {f.name} ({size:.0f}KB)")

print(f"\n{'=' * 60}")
print("💡 DISCUSSION:")
print("   Watch the generated clips carefully.")
print("   Do they match what you imagined from the text prompts?")
print("   What surprised you? What disappointed you?")
print("   How would real footage of the same scenes look different?")

---
## ⏭️ Next Steps

Your video clips are ready! Next up:

- **04_The_Voice.ipynb** — Generate narration audio with Text-to-Speech
- **06_Assemble.ipynb** — Combine videos + audio + subtitles into the final video

Or experiment:
- Try a **different model** by changing `CHOSEN_MODEL` and re-running
- **Replace individual clips** with found footage and compare
- **Edit prompts.json** and regenerate specific scenes

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*